# Retrieval Pipeline — Hybrid Search + Reranking

Câu hỏi → Query Processing → BM25 + E5/Qdrant → RRF → Reranker → top-5 và nguồn PDF.

Dùng index từ notebook 03. Chọn kernel .venv, chạy từ trên xuống.


## Setup

In [1]:
import hashlib
import json
import re
import sys
from pathlib import Path

import torch
from qdrant_client import QdrantClient
from rank_bm25 import BM25Okapi


## Config

Dùng cùng model và index của notebook 03.

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.ingestion.textbook_embedder import E5Embedder

CHUNK_PATH = ROOT / "output/chunks/structure_aware_recursive.jsonl"
MODEL_NAME = "intfloat/multilingual-e5-small"
REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
TOP_K, CANDIDATE_K = 5, 20
torch.set_num_threads(4)
from src.retrieval.textbook_retrieval import CrossEncoderReranker, TextbookRetriever
PROFILE = json.loads((ROOT / "output/retrieval/structure_aware_recursive/manifest.json").read_text())

## Mở dữ liệu

Lấy nội dung đã lưu để BM25 và dense tìm trên cùng tập chunk.

In [3]:
rows = [json.loads(line) for line in CHUNK_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
point_ids = [i for i, row in enumerate(rows) if row["metadata"]["sources"]]
key = MODEL_NAME + REVISION + CHUNK_PATH.read_text(encoding="utf-8")
collection = "e5_" + hashlib.sha256(key.encode()).hexdigest()[:12]
if "client" in globals():
    client.close()
client = QdrantClient(path=str(ROOT / "output/qdrant_db"))
points = client.retrieve(collection, ids=point_ids, with_payload=True)
points_by_id = {point.id: point for point in points}
texts = [points_by_id[i].payload["page_content"] for i in point_ids]
assert client.count(collection).count == len(point_ids)
assert all(points_by_id[i].payload == rows[i] for i in point_ids)

## Chuẩn bị BM25 và E5

Load một lần; vector tài liệu đã có trong Qdrant.

In [4]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

bm25 = BM25Okapi([tokenize(text) for text in texts])
embedder = E5Embedder(model_name=MODEL_NAME, revision=REVISION)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Query Processing

- Query Processing ở mức cơ bản là nhúng câu hỏi vào Embedding model. Trong tương lai xử lý multi-query và các bài toán phức tạp hơn

BM25 dùng tokenizer ở bước 6; E5Embedder thêm query prefix khi tạo vector.

In [5]:
QUERY = "What is a database schema?"
query_vector = embedder.embed_query(QUERY)

## Hybrid Search: lấy ứng viên

BM25 tìm theo từ khóa, E5/Qdrant tìm theo ngữ nghĩa. Mỗi nhánh lấy tối đa 20 chunk; BM25 chỉ giữ điểm dương.


In [6]:
scores = bm25.get_scores(tokenize(QUERY))
order = sorted(range(len(texts)), key=lambda i: (-scores[i], point_ids[i]))
lexical_ids = [point_ids[i] for i in order if scores[i] > 0][:CANDIDATE_K]

response = client.query_points(collection_name=collection, query=query_vector.tolist(),
                               limit=CANDIDATE_K, with_payload=False)
dense_ids = [point.id for point in sorted(response.points, key=lambda p: (-p.score, p.id))]

## Hybrid Search: gộp RRF

Gộp thứ hạng bằng 1 / (60 + rank), giữ 20 ứng viên để reranker chấm lại.


In [7]:
rrf_scores = {}
for ids in [lexical_ids, dense_ids]:
    for rank, point_id in enumerate(ids, 1):
        rrf_scores[point_id] = rrf_scores.get(point_id, 0) + 1 / (60 + rank)

candidate_ids = sorted(rrf_scores, key=lambda i: (-rrf_scores[i], i))[:CANDIDATE_K]

## Load reranker

Load model một lần, dùng chung cho các cell thực nghiệm và class. Model được lưu trong cache dự án.

Kiểm tra cấu hình embedding theo manifest, cùng nội dung và nguồn trong collection trước khi truy xuất.

In [8]:
reranker = CrossEncoderReranker(
    revision="953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e",
    cache_folder=str(ROOT / "output/model_cache/reranker_hub"),
)
retriever = TextbookRetriever(client, collection, embedder, reranker, embedding_profile=PROFILE)

The Transformer `cache_dir` argument is deprecated. Please pass `cache_dir` via `model_kwargs`, `processor_kwargs`, and/or `config_kwargs` instead.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

d:\Users\BuiTrongNguyen\CodexProject\db-rag-eval\src\retrieval\textbook_retrieval.py:119: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.embedder.model.get_sentence_embedding_dimension()


### Kiểm tra độ dài cặp câu hỏi–chunk

Đếm cả câu hỏi, chunk và special tokens. Báo lỗi nếu vượt giới hạn, để tránh cắt mất nội dung khi chấm điểm.

In [9]:
pairs = [[QUERY, points_by_id[i].payload["page_content"]] for i in candidate_ids]
for question, passage in pairs:
    count = len(reranker.model.tokenizer(question, passage, truncation=False)["input_ids"])
    if count > reranker.max_length:
        raise ValueError(f"Cặp query–chunk có {count} token, vượt {reranker.max_length}")

## Reranking

Chấm lại 20 cặp câu hỏi–chunk, rồi sắp xếp theo điểm mới. RRF dùng thứ hạng hai nhánh; reranker dùng trực tiếp nội dung từng cặp.


In [10]:
rerank_scores = reranker.model.predict(
    pairs, batch_size=2, show_progress_bar=False, activation_fn=torch.nn.Identity()
)
ranked_hits = sorted(zip(candidate_ids, rerank_scores), key=lambda hit: (-hit[1], hit[0]))

## Xem kết quả

Hiện top-5 sau reranking: nội dung, điểm reranker và trang PDF. Điểm cao hơn là liên quan hơn, không phải xác suất đúng. Đây là kết quả truy xuất để đưa vào bước tạo câu trả lời.


In [11]:
for rank, (point_id, score) in enumerate(ranked_hits[:TOP_K], 1):
    chunk = points_by_id[point_id].payload
    pages = sorted({source["pdf_page"] + 1 for source in chunk["metadata"]["sources"]})
    print(f"{rank}. Điểm reranker: {score:.4f} | Trang PDF: {pages}")
    print(chunk["page_content"], "\n")



1. Điểm reranker: 5.3008 | Trang PDF: [65]
[[SECTION 2.1.2: Schemas, Instances, and Database State]]
In a data model, it is important to distinguish between the description of the database and the database itself. The description of a database is called the database schema, which is specified during database design and is not expected to change frequently. Most data models have certain conventions for displaying schemas as diagrams. A displayed schema is called a schema diagram. Figure 2.1 shows a schema diagram for the database shown in Figure 1.2; the diagram displays the structure of each record type but not the actual instances of records.
Section_identifier Semester Course_number Instructor Year
SECTION
Course_name Course_number Credit_hours Department
COURSE
Name Student_number Class Major
STUDENT
Course_number Prerequisite_number PREREQUISITE
Student_number Grade Section_identifier
GRADE_REPORT
Figure 2.1 Schema diagram for the database in Figure 1.2. 

2. Điểm reranker: 2.6903 

## Gọi class và đối chiếu

Dùng lại model đã load. Kết quả từ class phải khớp ID, điểm, nội dung và nguồn của pipeline từng bước ở trên.

In [12]:
results = retriever.retrieve(QUERY, top_k=TOP_K, candidate_k=CANDIDATE_K)
assert [hit["point_id"] for hit in results] == [i for i, _ in ranked_hits[:TOP_K]]
for hit, (point_id, score) in zip(results, ranked_hits[:TOP_K]):
    assert abs(hit["score"] - float(score)) < 1e-5
    assert hit["page_content"] == points_by_id[point_id].payload["page_content"]
    assert hit["metadata"] == points_by_id[point_id].payload["metadata"]
print("Notebook và class khớp top-k, điểm và nguồn.")
client.close()

Notebook và class khớp top-k, điểm và nguồn.
